# Evaluación 3: Monte Carlo del momento de inercia de un disco
Ejecuta las celdas **en orden**. Antes de empezar: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*.
Los resultados se guardan en `Evaluacion3/nivel1`, `nivel2` y `nivel3`. La última celda los descarga en un zip.

## 0. Revisar que hay GPU

In [ ]:
!nvidia-smi
from numba import cuda
print("GPU CUDA disponible:", cuda.is_available())

## Nivel 1: versión secuencial validada

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

carpeta1 = Path("Evaluacion3/nivel1")
carpeta1.mkdir(parents=True, exist_ok=True)

rho0 = 1.0
R = 1.0
I_exacto = 0.9 * np.pi * rho0 * R**4


def monte_carlo(N):
    """Estima I con N puntos uniformes en el cuadrado [-R,R]x[-R,R]."""
    x = np.random.uniform(-R, R, N)
    y = np.random.uniform(-R, R, N)
    r = np.sqrt(x**2 + y**2)
    rho = rho0 * (1 + r / R)
    f = rho * r**2 * (r <= R)          # (r <= R) es la funcion indicadora
    A = (2 * R) ** 2                   # area del cuadrado
    I = A * np.mean(f)                 # estimador: area * promedio
    error_estadistico = A * np.std(f) / np.sqrt(N)
    return I, error_estadistico


Ns = [10**k for k in range(3, 8)]      # 10^3 ... 10^7
repeticiones = 20                      # promediamos el error para que la curva sea suave

print(f"I exacto = {I_exacto:.6f}\n")
print(f"{'N':>10} {'I (1 corrida)':>15} {'error abs.':>12} {'error estad.':>13} {'error medio(20)':>16}")

filas1 = []
errores_medios = []
for N in Ns:
    I, err_est = monte_carlo(N)
    err_abs = abs(I - I_exacto)
    errs = [abs(monte_carlo(N)[0] - I_exacto) for _ in range(repeticiones)]
    errores_medios.append(np.mean(errs))
    filas1.append([N, I, err_abs, err_est, errores_medios[-1]])
    print(f"{N:>10} {I:>15.6f} {err_abs:>12.2e} {err_est:>13.2e} {errores_medios[-1]:>16.2e}")

pendiente = np.polyfit(np.log10(Ns), np.log10(errores_medios), 1)[0]
print(f"\nPendiente ajustada en log-log: {pendiente:.2f} (esperada: -0.5)")

np.savetxt(carpeta1 / "resultados_nivel1.csv", filas1, delimiter=",",
           header="N,I,error_abs,error_estadistico,error_medio_20", comments="")

Ns_arr = np.array(Ns, dtype=float)
referencia = errores_medios[0] * (Ns_arr / Ns_arr[0]) ** (-0.5)

plt.loglog(Ns_arr, errores_medios, "o-", label="error medio de Monte Carlo")
plt.loglog(Ns_arr, referencia, "--", label="recta de pendiente -1/2")
plt.xlabel("N (numero de puntos)")
plt.ylabel("error absoluto")
plt.title("Convergencia del error")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.savefig(carpeta1 / "convergencia.png", dpi=150)
plt.show()

**Comentario (escríbelo con tus palabras):** ¿el error decae como 1/√N? Usa la pendiente ajustada que imprimió la celda anterior.

## Nivel 2: paralelización en CPU con Numba

In [ ]:
import os
import time
from numba import njit, prange

carpeta2 = Path("Evaluacion3/nivel2")
carpeta2.mkdir(parents=True, exist_ok=True)


@njit
def mc_secuencial(N):
    suma = 0.0
    for i in range(N):
        x = np.random.uniform(-1.0, 1.0)
        y = np.random.uniform(-1.0, 1.0)
        r2 = x * x + y * y
        if r2 <= 1.0:                       # funcion indicadora
            r = np.sqrt(r2)
            suma += (1.0 + r) * r2          # rho * r^2
    return 4.0 * suma / N                   # A = 4


@njit(parallel=True)
def mc_paralelo(N):
    suma = 0.0
    for i in prange(N):                     # las iteraciones se reparten entre los nucleos
        x = np.random.uniform(-1.0, 1.0)
        y = np.random.uniform(-1.0, 1.0)
        r2 = x * x + y * y
        if r2 <= 1.0:
            r = np.sqrt(r2)
            suma += (1.0 + r) * r2          # Numba suma las parciales de cada hilo
    return 4.0 * suma / N


def medir(funcion, N, repeticiones=3):
    """Devuelve (mejor tiempo, resultado)."""
    mejor = 1e99
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        resultado = funcion(N)
        t1 = time.perf_counter()
        mejor = min(mejor, t1 - t0)
    return mejor, resultado


# Calentamiento: la primera llamada compila (JIT); no se mide
mc_secuencial(1000)
mc_paralelo(1000)

Ns2 = [10**k for k in range(4, 9)]         # 10^4 ... 10^8
nucleos = os.cpu_count()
print(f"Nucleos de la maquina: {nucleos}\n")
print(f"{'N':>10} {'T sec (s)':>12} {'T par (s)':>12} {'Speedup':>9} {'I sec':>10} {'I par':>10}")

filas2 = []
for N in Ns2:
    t_sec, I_sec = medir(mc_secuencial, N)
    t_par, I_par = medir(mc_paralelo, N)
    S = t_sec / t_par
    filas2.append([N, t_sec, t_par, S, I_sec, I_par])
    print(f"{N:>10} {t_sec:>12.5f} {t_par:>12.5f} {S:>9.2f} {I_sec:>10.5f} {I_par:>10.5f}")

print(f"\nI exacto = {I_exacto:.5f}")

np.savetxt(carpeta2 / "resultados_nivel2.csv", filas2, delimiter=",",
           header="N,t_sec,t_par,speedup,I_sec,I_par", comments="")
with open(carpeta2 / "nucleos.txt", "w") as f:
    f.write(f"Nucleos de la maquina: {nucleos}\n")

arr2 = np.array(filas2)
plt.semilogx(arr2[:, 0], arr2[:, 3], "o-", label="CPU paralela (Numba)")
plt.axhline(1.0, color="gray", linestyle="--", label="speedup = 1")
plt.xlabel("N")
plt.ylabel("Speedup  S = T_sec / T_par")
plt.title(f"Speedup en CPU ({nucleos} nucleos)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(carpeta2 / "speedup_cpu.png", dpi=150)
plt.show()

**Comentario (escríbelo con tus palabras):** ¿por qué el speedup es bajo, o menor que 1, para N pequeño?

## Nivel 3: paralelización en GPU con numba.cuda

In [ ]:
import math
from numba import cuda
from numba.cuda.random import create_xoroshiro128p_states, xoroshiro128p_uniform_float64

carpeta3 = Path("Evaluacion3/nivel3")
carpeta3.mkdir(parents=True, exist_ok=True)

HILOS_POR_BLOQUE = 256
MAX_HILOS = 100_000          # maximo de hilos que usamos


@cuda.jit
def kernel(rng_states, muestras_por_hilo, parciales):
    i = cuda.grid(1)                              # indice global del hilo
    if i < parciales.size:
        suma = 0.0
        for k in range(muestras_por_hilo):
            # numero uniforme en [0,1) de SU estado -> lo llevamos a [-1,1)
            x = 2.0 * xoroshiro128p_uniform_float64(rng_states, i) - 1.0
            y = 2.0 * xoroshiro128p_uniform_float64(rng_states, i) - 1.0
            r2 = x * x + y * y
            if r2 <= 1.0:
                r = math.sqrt(r2)
                suma += (1.0 + r) * r2
        parciales[i] = suma                       # suma parcial de este hilo


def mc_gpu(N, rng_states, n_hilos, d_parciales):
    muestras_por_hilo = N // n_hilos
    bloques = (n_hilos + HILOS_POR_BLOQUE - 1) // HILOS_POR_BLOQUE
    kernel[bloques, HILOS_POR_BLOQUE](rng_states, muestras_por_hilo, d_parciales)
    cuda.synchronize()
    parciales = d_parciales.copy_to_host()
    return 4.0 * parciales.sum() / N              # reduccion final en CPU


def medir_gpu(N, repeticiones=3):
    n_hilos = min(N, MAX_HILOS)
    rng_states = create_xoroshiro128p_states(n_hilos, seed=12345)   # un estado por hilo
    d_parciales = cuda.device_array(n_hilos, dtype=np.float64)
    mc_gpu(N, rng_states, n_hilos, d_parciales)                     # calentamiento (compila)
    mejor = 1e99
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        I = mc_gpu(N, rng_states, n_hilos, d_parciales)
        mejor = min(mejor, time.perf_counter() - t0)
    return mejor, I


def sigma_I(N):
    """Error estadistico de I para N puntos (estimado con una muestra en CPU)."""
    x = np.random.uniform(-1, 1, 10**6)
    y = np.random.uniform(-1, 1, 10**6)
    r = np.sqrt(x**2 + y**2)
    f = (1 + r) * r**2 * (r <= 1)
    return 4.0 * np.std(f) / math.sqrt(N)


print(f"{'N':>10} {'T gpu (s)':>11} {'I gpu':>10} {'|I-Iexacto|':>12} {'3*sigma':>10} {'OK?':>5} {'S gpu':>9}")
filas3 = []
speedup_gpu = []
for N, t_sec, t_par, S_par, I_sec, I_par in filas2:     # usa los tiempos de CPU de esta misma maquina
    N = int(N)
    t_gpu, I_gpu = medir_gpu(N)
    err = abs(I_gpu - I_exacto)
    tres_sigma = 3 * sigma_I(N)
    ok = "si" if err < tres_sigma else "no"
    S = t_sec / t_gpu
    speedup_gpu.append(S)
    filas3.append([N, t_sec, t_par, t_gpu, S_par, S, I_gpu, err, tres_sigma])
    print(f"{N:>10} {t_gpu:>11.5f} {I_gpu:>10.5f} {err:>12.2e} {tres_sigma:>10.2e} {ok:>5} {S:>9.2f}")

np.savetxt(carpeta3 / "resultados_nivel3.csv", filas3, delimiter=",",
           header="N,t_sec,t_par,t_gpu,speedup_cpu,speedup_gpu,I_gpu,error_abs,tres_sigma",
           comments="")

plt.semilogx(arr2[:, 0], arr2[:, 3], "o-", label="CPU paralela")
plt.semilogx(arr2[:, 0], speedup_gpu, "s-", label="GPU (CUDA)")
plt.axhline(1.0, color="gray", linestyle="--")
plt.xlabel("N")
plt.ylabel("Speedup respecto a CPU secuencial")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(carpeta3 / "speedup_total.png", dpi=150)
plt.show()

## Descargar los resultados

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader > Evaluacion3/nivel3/gpu_usada.txt
!zip -r Evaluacion3.zip Evaluacion3
from google.colab import files
files.download("Evaluacion3.zip")